# Gensim: LDA Topic Modeling using Gensim

In [1]:
%load_ext autoreload
%autoreload 2

import logging
import Gensim_topic_modeling_utils as topic_utils 


logging.basicConfig(level=logging.INFO)
_LOG = logging.getLogger(__name__)

_LOG.info("Setup complete")

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:__main__:Setup complete


In [2]:
import pandas as pd
import numpy as np

## Text Pre-processing

Before topic modeling, the text must be cleaned and converted into tokens which makes it easier for the model to understand the words

- **Tokenization**: split text into individual words
- **Stopword removal**: remove common words like "the", "is"
- **Lemmatization**: reduce words to base form ("running" → "run")

In [3]:
topic_utils.download_nltk_data()

In [4]:
sample_df = topic_utils.sample_dataset()
sample_df.head()

,title,content,topic_actual
0,ASIAN EXPORTERS FEAR DAMAGE FROM U.S.-JAPAN RIFT,Mounting trade friction between the U.S. And...,trade
1,THAI TRADE DEFICIT WIDENS IN FIRST QUARTER,Thailand's trade deficit widened to 4.5 bill...,trade
2,JAPAN MINISTRY SAYS OPEN FARM TRADE WOULD HIT ...,"Japan's Agriculture Ministry, angered by U.S...",trade
3,SOUTH KOREA MOVES TO SLOW GROWTH OF TRADE SURPLUS,South Korea's trade surplus is growing too f...,trade
4,JAPAN GIVEN LITTLE HOPE OF AVOIDING U.S. SANCT...,A top U.S. Official said Japan has little ch...,trade


In [5]:
sample_df.shape

(43, 3)

In [6]:
# Preprocess all documents 
processed_docs = topic_utils.preprocess_corpus(list(sample_df['content']))

print("Output tokens of the first sample document: \n",processed_docs[0])

Output tokens of the first sample document: 
 ['mounting', 'trade', 'friction', 'japan', 'raised', 'fear', 'among', 'many', 'asia', 'exporting', 'nation', 'row', 'could', 'inflict', 'economic', 'damage', 'businessmen', 'official', 'said', 'told', 'reuter', 'correspondent', 'asian', 'capital', 'move', 'japan', 'might', 'boost', 'protectionist', 'sentiment', 'lead', 'curb', 'american', 'import', 'product', 'exporter', 'said', 'conflict', 'would', 'hurt', 'tokyo', 'loss', 'might', 'gain', 'said', 'impose', 'mln', 'dlrs', 'tariff', 'import', 'japanese', 'electronics', 'good', 'april', 'retaliation', 'japan', 'alleged', 'failure', 'stick', 'pact', 'sell', 'semiconductor', 'world', 'market', 'cost', 'unofficial', 'japanese', 'estimate', 'put', 'impact', 'tariff', 'billion', 'dlrs', 'spokesman', 'major', 'electronics', 'firm', 'said', 'would', 'virtually', 'halt', 'export', 'product', 'hit', 'new', 'tax', 'would', 'able', 'business', 'said', 'spokesman', 'leading', 'japanese', 'electronics', 

## Word dictionary and corpus

Gensim LDA requires two structures:

- **Dictionary**: maps every unique token to an integer ID
- **Corpus**: converts each document into a bag-of-words (BoW) vector, i.e. a list of `(token_id, count)` pairs

Filtering the dictionary removes tokens that are too rare or too common,
keeping only the most informative vocabulary.


In [7]:
# Build dictionary from processed documents
# no_below=2: token must appear in at least 2 docs
# no_above=0.85: token must not appear in more than 85% of docs
# keep_n=500: hard cap on vocabulary size
dictionary = topic_utils.build_dictionary(
    processed_docs,
    no_below=2,
    no_above=0.85,
    keep_n=500
)

# Convert each document to BoW representation
corpus = topic_utils.build_corpus(processed_docs, dictionary)

# Inspect BoW for first document
print("BoW for doc 1: \n", corpus[0])
# Each tuple is (token_id, count)

INFO:gensim.corpora.dictionary:adding document #0 to Dictionary<0 unique tokens: []>
INFO:gensim.corpora.dictionary:built Dictionary<1673 unique tokens: ['able', 'accounting', 'action', 'advantage', 'alleged']...> from 43 documents (total 6148 corpus positions)
INFO:gensim.utils:Dictionary lifecycle event {'msg': "built Dictionary<1673 unique tokens: ['able', 'accounting', 'action', 'advantage', 'alleged']...> from 43 documents (total 6148 corpus positions)", 'datetime': '2026-05-06T21:15:05.100602', 'gensim': '4.4.0', 'python': '3.12.13 (main, Apr 22 2026, 02:12:20) [GCC 14.2.0]', 'platform': 'Linux-6.12.76-linuxkit-aarch64-with-glibc2.41', 'event': 'created'}
INFO:gensim.corpora.dictionary:discarding 1173 tokens: [('accounting', 1), ('advantage', 1), ('alleged', 1), ('asian', 1), ('australian', 1), ('avowed', 1), ('awaiting', 1), ('aware', 1), ('beyond', 1), ('biggest', 1)]...
INFO:gensim.corpora.dictionary:keeping 500 tokens which were in no less than 2 and no more than 36 (=85.0%) 

Vocabulary before filtering : 1673
Vocabulary after filtering  : 500
BoW for doc 1: 
 [(0, 1), (1, 1), (2, 2), (3, 4), (4, 1), (5, 2), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 2), (12, 1), (13, 1), (14, 5), (15, 1), (16, 2), (17, 1), (18, 1), (19, 1), (20, 1), (21, 2), (22, 2), (23, 1), (24, 1), (25, 1), (26, 3), (27, 1), (28, 1), (29, 1), (30, 2), (31, 1), (32, 1), (33, 1), (34, 2), (35, 6), (36, 1), (37, 2), (38, 2), (39, 1), (40, 1), (41, 1), (42, 1), (43, 1), (44, 7), (45, 1), (46, 1), (47, 1), (48, 2), (49, 1), (50, 1), (51, 1), (52, 1), (53, 1), (54, 1), (55, 2), (56, 1), (57, 1), (58, 1), (59, 1), (60, 1), (61, 1), (62, 6), (63, 1), (64, 1), (65, 3), (66, 1), (67, 1), (68, 1), (69, 12), (70, 4), (71, 1), (72, 3), (73, 1), (74, 1), (75, 2), (76, 4), (77, 1), (78, 2), (79, 1), (80, 1), (81, 3), (82, 1), (83, 2), (84, 1), (85, 1), (86, 1), (87, 3), (88, 1), (89, 3), (90, 1), (91, 1), (92, 3), (93, 1), (94, 1), (95, 1), (96, 1), (97, 1), (98, 1), (99, 1), (100, 1), (101, 1), (1

## Coherence Evaluation and Topic Selection

Choosing the right number of topics is critical. We use the **c_v coherence score** which measures how semantically similar the top words in each topic are.

- Higher coherence → more interpretable topics
- We train models for a range of topic counts and pick the one with the
  highest coherence score
- Look for a clear **peak** or **elbow** in the coherence curve

In [8]:
# Compute coherence for topic counts 2 through 6

model_list, coherence_values = topic_utils.compute_coherence_values(
    dictionary = dictionary,
    corpus     = corpus,
    docs    = processed_docs,
    start      = 2,
    limit      = 7,
    step       = 1
)

# Coherence scores for each topic size 

score_dict = {}
for i, cv in enumerate(coherence_values):
    score_dict[i+2] = cv
    print(f"Num topics: {i+2:2d} | Coherence: {cv:.4f}")

INFO:gensim.models.ldamodel:using autotuned alpha, starting with [np.float32(0.5), np.float32(0.5)]
INFO:gensim.models.ldamodel:using symmetric eta at 0.5
INFO:gensim.models.ldamodel:using serial LDA version on this node
INFO:gensim.models.ldamodel:running online (multi-pass) LDA training, 2 topics, 15 passes over the supplied corpus of 43 documents, updating model once every 43 documents, evaluating perplexity every 43 documents, iterating 50x with a convergence threshold of 0.001000
INFO:gensim.models.ldamodel:-6.662 per-word bound, 101.3 perplexity estimate based on a held-out corpus of 43 documents with 4431 words
INFO:gensim.models.ldamodel:PROGRESS: pass 0, at document #43/43
INFO:gensim.models.ldamodel:optimized alpha [np.float32(0.70158947), np.float32(0.79242766)]
INFO:gensim.models.ldamodel:topic #0 (0.702): 0.052*"said" + 0.042*"pct" + 0.017*"year" + 0.016*"february" + 0.016*"japan" + 0.014*"january" + 0.014*"sale" + 0.011*"market" + 0.011*"dlrs" + 0.010*"billion"
INFO:gensi

Num topics:  2 | Coherence: 0.3625
Num topics:  3 | Coherence: 0.4541
Num topics:  4 | Coherence: 0.4483
Num topics:  5 | Coherence: 0.4024
Num topics:  6 | Coherence: 0.4228


## LDA Model Training

Latent Dirichlet Allocation (LDA) is a generative probabilistic model that discovers hidden topics in a corpus. Each document is modelled as a mixture of topics, and each topic is modelled as a mixture of words.

Key parameters:
- **num_topics**: number of topics to discover
- **passes**: number of full training passes over the corpus (more = better convergence)
- **iterations**: number of E-step iterations per document per pass
- **alpha**: document-topic distribution prior (`'auto'` = learned from data)
- **eta**: topic-word distribution prior (`'auto'` = learned from data)

In [9]:
# Train LDA model with 3 topics on sample data
lda_model = topic_utils.train_lda_model(
    corpus     = corpus,
    dictionary = dictionary,
    num_topics = 3,
    passes     = 10,      
    iterations = 100
)

# Print top 5 words per topic
for idx, topic in lda_model.print_topics(num_words=5):
    _LOG.info("Topic %d: %s", idx, topic)

INFO:gensim.models.ldamodel:using autotuned alpha, starting with [np.float32(0.33333334), np.float32(0.33333334), np.float32(0.33333334)]
INFO:gensim.models.ldamodel:using serial LDA version on this node
INFO:gensim.models.ldamodel:running online (multi-pass) LDA training, 3 topics, 10 passes over the supplied corpus of 43 documents, updating model once every 43 documents, evaluating perplexity every 0 documents, iterating 100x with a convergence threshold of 0.001000
INFO:gensim.models.ldamodel:PROGRESS: pass 0, at document #43/43
INFO:gensim.models.ldamodel:optimized alpha [np.float32(0.31482893), np.float32(0.39284253), np.float32(0.26771745)]
INFO:gensim.models.ldamodel:topic #0 (0.315): 0.054*"said" + 0.036*"pct" + 0.020*"year" + 0.019*"japan" + 0.014*"trade" + 0.013*"market" + 0.011*"japanese" + 0.010*"february" + 0.009*"dlrs" + 0.009*"billion"
INFO:gensim.models.ldamodel:topic #1 (0.393): 0.043*"pct" + 0.032*"said" + 0.026*"sale" + 0.025*"trade" + 0.023*"billion" + 0.020*"dlrs" 

LDA model trained with 3 topics


## Topic Inspection and Labelling

After training, topics need to be inspected manually as LDA only clusters the documents into topics but doesn't provide human understandable labels. Topic inspection here includes inspecting top words and representative documents per topic, then manually assign human-readable labels.

Workflow:
- `inspect_topics()`: read output, decide labels
- Define `topic_labels` dict as `{topic_id: label}`
- `assign_topics()`: adds columns to dataframe

In [10]:
# Step 1 — inspect topics, no labels needed yet
topic_utils.inspect_topics(
    model       = lda_model,
    corpus      = corpus,
    df   = sample_df,
    title_column='title',
    num_topics  = 3,
    num_words   = 5,
    num_docs    = 5
)

  TOPIC 0

  Top 5 words:
    0.0570  - said
    0.0326  - japan
    0.0284  - trade
    0.0217  - market
    0.0163  - japanese

  Top 5 representative articles:
    1. [1.00] ECONOMIC SPOTLIGHT - TELECOM IS KEY JAPAN MINISTRY
    2. [1.00] JAPAN MINISTRY SAYS OPEN FARM TRADE WOULD HIT U.S.
    3. [1.00] JAPAN IN LAST-DITCH EFFORT TO AVERT TARIFFS
    4. [1.00] JAPANESE OFFICIAL TAKES DATA TO MICROCHIP TALKS
    5. [1.00] CANADA TO MONITOR STEEL IMPORTS, EXPORTS

  TOPIC 1

  Top 5 words:
    0.0308  - year
    0.0294  - billion
    0.0269  - said
    0.0269  - dlrs
    0.0256  - trade

  Top 5 representative articles:
    1. [1.00] DEFICIT CUTS SEEN UNABLE TO CURE TRADE DEFICIT
    2. [1.00] SOUTH KOREA MOVES TO SLOW GROWTH OF TRADE SURPLUS
    3. [1.00] JAPAN GIVEN LITTLE HOPE OF AVOIDING U.S. SANCTIONS
    4. [1.00] JAPAN GIVEN LITTLE HOPE OF AVOIDING U.S. SANCTIONS
    5. [1.00] G-7 ISSUES STATEMENT AFTER MEETING

  TOPIC 2

  Top 5 words:
    0.0851  - pct
    0.0555  - sale
    

In [11]:
topic_labels = {
    0: 'Tech',
    1: 'Trade',
    2: 'Retail'
}

In [12]:
sample_df = topic_utils.assign_topics(
    df           = sample_df,
    corpus       = corpus,
    model        = lda_model,
    topic_labels = topic_labels
)

In [13]:
sample_df.head()

,title,content,topic_actual,cluster_id,topic_confidence,topic_label
0,ASIAN EXPORTERS FEAR DAMAGE FROM U.S.-JAPAN RIFT,Mounting trade friction between the U.S. And...,trade,0,0.7787,Tech
1,THAI TRADE DEFICIT WIDENS IN FIRST QUARTER,Thailand's trade deficit widened to 4.5 bill...,trade,2,0.9986,Retail
2,JAPAN MINISTRY SAYS OPEN FARM TRADE WOULD HIT ...,"Japan's Agriculture Ministry, angered by U.S...",trade,0,0.9992,Tech
3,SOUTH KOREA MOVES TO SLOW GROWTH OF TRADE SURPLUS,South Korea's trade surplus is growing too f...,trade,1,0.9993,Trade
4,JAPAN GIVEN LITTLE HOPE OF AVOIDING U.S. SANCT...,A top U.S. Official said Japan has little ch...,trade,1,0.9986,Trade


## Topic Visualization with pyLDAvis

pyLDAvis provides an interactive visualization of the LDA model with two panels:

- **Left panel** — intertopic distance map. Each bubble is a topic, size
  represents prevalence, distance represents similarity between topics
- **Right panel** — top 30 most relevant terms for the selected topic

Interpretation
- Well-separated bubbles: distinct, high-quality topics
- Overlapping bubbles: topics too similar 
- Setting **λ = 0.6** shows the most meaningful word rankings

In [14]:
topic_utils.get_pyldavis_visualization(lda_model, corpus, dictionary)

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0      0.098046  0.113493       1        1  26.854909
1      0.081083 -0.088529       2        1  25.827989
2     -0.179129 -0.024964       3        1  47.317102, topic_info=         Term        Freq       Total Category  logprob  loglift
135      sale  122.000000  122.000000  Default  30.0000  30.0000
447   january   76.000000   76.000000  Default  29.0000  29.0000
111       pct  213.000000  213.000000  Default  28.0000  28.0000
403  february   77.000000   77.000000  Default  27.0000  27.0000
69      japan   49.000000   49.000000  Default  26.0000  26.0000
..        ...         ...         ...      ...      ...      ...
171      year   26.503464   68.792570   Topic3  -4.3708  -0.2055
35       dlrs   22.237442   59.034650   Topic3  -4.5463  -0.2280
332      data   11.606922   16.589756   Topic3  -5.1965   0.3911
233      rise   12.365140   21.584484   Topic3  -5.1332   0.1912
76       last   13.355711   30.343493   Topic3  -5.0561  -0.0723

[195 rows x 6 columns], token_table=      Topic      Freq       Term
term                            
456       3  0.892602  according
1         1  0.318860     action
1         2  0.744007     action
448       3  0.970587   adjusted
204       1  0.903106    already
...     ...       ...        ...
170       2  0.420421      would
170       3  0.152880      would
171       1  0.101755       year
171       2  0.508776       year
171       3  0.392484       year

[264 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2, 3])